# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading, inspecting, and processing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema, available via a public URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a native object (not as dict)
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
if getattr(dataset.metadata, 'keywords', None):
    print(f"Keywords: {', '.join(dataset.metadata.keywords)}")

## 2. Data Overview
Let's review the available record sets, fields, and their unique `@id`s as defined by the Croissant schema. These `@id`s are used for precise references in Croissant.

> **Note:** A record set in Croissant bundles together a data table and schema description.

In [ ]:
# List all available record sets and their @ids
if hasattr(dataset.metadata, 'recordSets') and dataset.metadata.recordSets:
    print("Record sets in dataset:")
    for rs in dataset.metadata.recordSets:
        print(f" - Name: {getattr(rs, 'name', None)} | @id: {rs.id}")
else:
    # For this FAIR^2 schema, we get the main record set from distribution
    print("No explicit 'recordSets' in metadata. Checking main tables...")
    if hasattr(dataset.metadata, 'distribution') and dataset.metadata.distribution:
        dist = dataset.metadata.distribution[0]
        if hasattr(dist, 'recordSet') and dist.recordSet:
            print("Record sets from distribution:")
            for rs in dist.recordSet:
                print(f" - Name: {getattr(rs, 'name', None)} | @id: {rs.id}")
        else:
            print("No record sets found in both metadata and distribution.")
    else:
        print("No record sets or distribution information found.")

In [ ]:
# When record sets not directly accessible through metadata,
# mlcroissant exposes them via dataset.record_sets

record_sets = [rs['@id'] for rs in dataset.record_sets]
print(f"Available record set @ids: {record_sets}")

# Display each record set's fields and columns by @id
for recset in dataset.record_sets:
    print(f"\nRecord Set: {recset['name']}")
    print(f"  @id: {recset['@id']}")
    print("Fields:")
    for field in recset['fields']:
        print(f"    - {field['name']} | @id: {field['@id']}")
    print("Columns:")
    for col in recset['columns']:
        print(f"    - {col['name']} | @id: {col['@id']}")

## 3. Data Extraction
We will extract data from the main record set into a pandas DataFrame for further analysis.

Record sets and fields are always referenced by their `@id`.


In [ ]:
# Extract data from all record sets
# (We use @id for reference, as per the Croissant schema)
dataframes = {}

# We'll load all record sets listed
for record_set_id in record_sets:
    print(f"\nLoading records from record set {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")

# Pick the first record set @id for demonstration in EDA/visualization below
main_record_set_id = record_sets[0]
print(f"\nMain table columns in record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Now, let's explore and process the data. We'll show examples like filtering, normalization, and grouping using field `@id` names.

> For this dataset (protein data), choose a numeric field such as peptide count, abundance, coverage, or molecular weight (`MW`) by its `@id` from above. We'll use the first numeric field found.


In [ ]:
import numpy as np

# Let's attempt to auto-detect a numeric field (by scanning column names)
df = dataframes[main_record_set_id]

numeric_field_id = None
candidate_keywords = ['count', 'abundance', 'MW', 'coverage', 'peptide', 'intensity', 'ratio', 'score', 'number']
for col in df.columns:
    if any(word.lower() in col.lower() for word in candidate_keywords):
        try:
            # Try converting to float, if succeeds use
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if df[col].notna().sum() >= 5:  # reasonable column
                numeric_field_id = col
                break
        except Exception:
            continue
if numeric_field_id is None:
    # Fallback: try the first float-like column
    numeric_field_id = df.select_dtypes(include=[np.number]).columns[0]

print(f"Using numeric field for analysis: {numeric_field_id}")

# Filtering step (e.g., abundance/count > threshold)
threshold = df[numeric_field_id].mean()
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered to {len(filtered_df)} records with {numeric_field_id} > {threshold:.2f}\n")

# Normalize field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a possible categorical variable (e.g. sampleType, accession, or other non-numeric field)
group_candidates = [col for col in df.columns if col != numeric_field_id and df[col].nunique() < 20]
group_field = group_candidates[0] if group_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field and (if grouping was possible) means across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

if group_field:
    plt.figure(figsize=(8,5))
    sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df)
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` for FAIR^2 dataset exploration:
- Accessing and loading dataset structure (with Croissant schema and `@id` references)
- Extracting available record sets and their fields, by `@id`
- Importing data into DataFrames and conducting typical EDA operations with proper identifiers
- Filtering, normalizing and grouping data, and visualizing core dataset metrics

You can extend this notebook for downstream analyses relevant to biological mass spectrometry data, machine learning feature engineering, or integration with additional omics resources.